# Newsletter Pipeline Observability

Concrete examples pulled from the SQLite DB and (optionally) your vault.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # notebook lives in observability/
sys.path.insert(0, str(repo_root))


In [ ]:
import os
import sqlite3
from textwrap import indent

try:
    import pandas as pd
except Exception:
    pd = None

def open_db(db_path: str) -> sqlite3.Connection:
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database not found: {db_path}")
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    return conn

def rows(conn: sqlite3.Connection, query: str, params=()):
    return [dict(r) for r in conn.execute(query, params).fetchall()]

def first_row(conn: sqlite3.Connection, query: str, params=()):
    row = conn.execute(query, params).fetchone()
    return dict(row) if row else None

def file_preview(path: str, max_lines: int = 40) -> str:
    if not path or not os.path.exists(path):
        return "<missing>"
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    return "".join(lines[:max_lines]).rstrip()

def show_table(data, title=None):
    if title:
        print(title)
    if not data:
        print("(no rows)")
        return
    if pd is not None:
        display(pd.DataFrame(data))
    else:
        for row in data:
            print(row)

# Parameters
db_path = "../newsletter.db"
vault_path = "../vault"  # optional, e.g. r"C:\\Path\\To\\Vault"
max_samples = 3

conn = open_db(db_path)


In [3]:
# Gmail examples
gmail_count = first_row(conn, "SELECT COUNT(*) AS n FROM gmail_messages")
print(f"Total messages: {gmail_count['n'] if gmail_count else 0}")
gmail_samples = rows(
    conn,
    """
    SELECT message_id, internal_date, subject, from_email, issue_note_path
    FROM gmail_messages
    ORDER BY internal_date DESC
    LIMIT ?
    """,
    (max_samples,),
)
show_table(gmail_samples, title="Recent messages")

for msg in gmail_samples:
    if msg.get("issue_note_path"):
        preview = file_preview(msg["issue_note_path"])
        print("\nIssue note preview:")
        print(indent(preview or "<empty>", "  "))


Total messages: 20
Recent messages


,message_id,internal_date,subject,from_email,issue_note_path
0,19c11e9465c23bd6,1769827026000,Robbyant Open Sources LingBot World | DeepSee...,Marktechpost AI <marktechpost-newsletter@mail....,C:\Users\felze\source\repos\newsletter\vault\N...
1,19c0f4bb1e82a873,1769783144000,"Apple audio AI acquisition 🎧, OpenAI’s data ag...",TLDR AI <dan@tldrnewsletter.com>,C:\Users\felze\source\repos\newsletter\vault\N...
2,19c0eda69526339e,1769775720000,"Google’s AI SRE 🔍, Sentry’s CLI 🆕, 10 Years of...",TLDR DevOps <dan@tldrnewsletter.com>,C:\Users\felze\source\repos\newsletter\vault\N...



Issue note preview:
  ---
  type: newsletter-issue
  source: "Robbyant Open Sources LingBot World |  DeepSeek AI Releases DeepSeek-OCR 2"
  date: 2026-01-30
  gmail_message_id: "19c11e9465c23bd6"
  from: "Marktechpost AI <marktechpost-newsletter@mail.beehiiv.com>"
  tags: [newsletters]
  ---

  # Summary
  - Total links: 67
  - Domains: 10

  # Links
  ## ainews.sh

  - [https://ainews.sh/ProductDetail?id=697c618bd71aafc3264b92fa](https://ainews.sh/ProductDetail?id=697c618bd71aafc3264b92fa)
  - [https://ainews.sh/ProductDetail?id=697d21c5a07d776f43e470cd](https://ainews.sh/ProductDetail?id=697d21c5a07d776f43e470cd)
  - [https://ainews.sh/ProductDetail?id=697d249dcb670de635f290b1](https://ainews.sh/ProductDetail?id=697d249dcb670de635f290b1)
  - [https://ainews.sh/ProductDetail?id=697c6bd05dcfa5082abae5f2](https://ainews.sh/ProductDetail?id=697c6bd05dcfa5082abae5f2)
  - [https://ainews.sh/ProductDetail?id=697d22bdca2a6348f98bdb32](https://ainews.sh/ProductDetail?id=697d22bdca2a6348f98bd

In [ ]:
# Link examples
link_count = first_row(conn, "SELECT COUNT(*) AS n FROM links")
print(f"Total links: {link_count['n'] if link_count else 0}")

status_counts = rows(
    conn,
    """
    SELECT COALESCE(fetch_status, 'unprocessed') AS status, COUNT(*) AS n
    FROM links
    GROUP BY status
    ORDER BY n DESC
    """,
)
show_table(status_counts, title="Status breakdown")

top_domains = rows(
    conn,
    """
    SELECT domain, COUNT(*) AS n
    FROM links
    GROUP BY domain
    ORDER BY n DESC
    LIMIT ?
    """,
    (max_samples,),
)
show_table(top_domains, title="Top domains")

link_samples = rows(
    conn,
    """
    SELECT url_canonical, domain, fetch_status, title, summary, note_path
    FROM links
    ORDER BY discovered_at DESC
    LIMIT ?
    """,
    (max_samples,),
)
show_table(link_samples, title="Recent links")

for link in link_samples:
    if link.get("summary"):
        print("\nSummary:")
        print(indent(link["summary"].strip(), "  "))
    if link.get("note_path"):
        preview = file_preview(link["note_path"])
        print("\nArticle note preview:")
        print(indent(preview or "<empty>", "  "))


## Web requests + parsing (pipeline demo)

The pipeline uses `fetch_article()` and `extract_main_text()` from `run.py`.
The cells below show how those methods behave on a single URL.
They do **live HTTP requests** when executed.

In [ ]:
from run import fetch_article, extract_main_text

# Pick any URL you want to test
test_url = "https://example.com"

status, title, text = fetch_article(test_url)
print("status:", status)
print("title:", title)
print("text preview:", (text or "")[:500])


In [ ]:
# Optional: if you already have HTML (e.g., saved response), parse it directly
html = "<html><head><title>Sample</title></head><body><h1>Hello</h1><p>World</p></body></html>"
title2, text2 = extract_main_text(html)
print("title:", title2)
print("text:", text2)


In [6]:
# Optional vault checks
if vault_path:
    issues_root = os.path.join(vault_path, "Newsletters", "Issues")
    articles_root = os.path.join(vault_path, "Newsletters", "Articles")
    print(f"Issues root exists: {issues_root} -> {os.path.exists(issues_root)}")
    print(f"Articles root exists: {articles_root} -> {os.path.exists(articles_root)}")
else:
    print("Set vault_path to preview files under your vault.")


Set vault_path to preview files under your vault.
